# Proyecto de sprint 8 - TripleTen

### 1. Extracción de datos de URL y creación de Data Frame con pandas

In [4]:
# Importar librerías
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL con los datos del clima
URL = 'https://practicum-content.s3.us-west-1.amazonaws.com/data-analyst-eng/moved_chicago_weather_2017.html'

# Hacer la petición GET para obtener el HTML
req = requests.get(URL)

# Parsear el contenido HTML con BeautifulSoup
soup = BeautifulSoup(req.text, 'lxml')

# Localizar la tabla por su atributo id ('weather_records')
table = soup.find('table', attrs={"id": "weather_records"})

# Extraer los encabezados de la tabla (etiquetas <th>)
heading_table = []
for row in table.find_all('th'):
    heading_table.append(row.text)

# Extraer las filas de datos (etiquetas <td> dentro de cada <tr>)
content = []
for row in table.find_all('tr'):
    if not row.find_all('th'):  # saltar la fila de encabezados
        content.append([element.text for element in row.find_all('td')])

# Crear el DataFrame con los datos y encabezados
weather_records = pd.DataFrame(content, columns=heading_table)

# Mostrar el resultado
print(weather_records)

ModuleNotFoundError: No module named 'bs4'

## Análisis exploratorio de datos

1. Recuperación de datos de cantidad de viajes por empresa (Usamos SQL)

In [ ]:
SELECT
    cabs.company_name AS company_name,
    COUNT(trips.trip_id) AS trips_amount
FROM
    cabs
    INNER JOIN trips ON trips.cab_id = cabs.cab_id
WHERE
    CAST(trips.start_ts AS date) BETWEEN '2017-11-15' AND '2017-11-16'
GROUP BY
    cabs.company_name
ORDER BY
    trips_amount DESC;

2. Encontramos cantidad de viajes para empresas con "blue" o "yellow" en su nombre

In [ ]:
SELECT
    cabs.company_name AS company_name,
    COUNT(trips.trip_id) AS trips_amount
FROM
    cabs
    INNER JOIN trips ON trips.cab_id = cabs.cab_id
WHERE
    (cabs.company_name LIKE '%Yellow%' OR cabs.company_name LIKE '%Blue%')
    AND CAST(trips.start_ts AS date) BETWEEN '2017-11-01' AND '2017-11-07'
GROUP BY
    cabs.company_name;

3. Obtenemos los números de las empresas más populares durante Nov de 2017 y agrupamos las demás como "other"

In [ ]:
SELECT
    CASE
        WHEN cabs.company_name = 'Flash Cab' THEN 'Flash Cab'
        WHEN cabs.company_name = 'Taxi Affiliation Services' THEN 'Taxi Affiliation Services'
        ELSE 'Other'
    END AS company,
    COUNT(trips.trip_id) AS trips_amount
FROM
    cabs
    INNER JOIN trips ON trips.cab_id = cabs.cab_id
WHERE
    CAST(trips.start_ts AS date) BETWEEN '2017-11-01' AND '2017-11-07'
GROUP BY
    company
ORDER BY
    trips_amount DESC;

4. Recuperamos identificadores de barrio para O'Hare y Loop

In [ ]:
SELECT
    neighborhood_id,
    name
FROM
    neighborhoods
WHERE
    name LIKE '%Hare%'
    OR name LIKE 'Loop%';

5. Obtenemos los registros por hora para catalogarlos en 2 categorías "Bad" & "Good"

In [ ]:
SELECT
    ts,
    CASE
        WHEN description LIKE '%rain%' OR description LIKE '%storm%' THEN 'Bad'
        ELSE 'Good'
    END AS weather_conditions
FROM
    weather_records;

6. Extraemos datos para los días domingos de un trayecto específico para ver la influencia del clima en la duración de los viajes

In [ ]:
SELECT
    trips.start_ts,
    CASE
        WHEN weather_records.description LIKE '%rain%' OR weather_records.description LIKE '%storm%' THEN 'Bad'
        ELSE 'Good'
    END AS weather_conditions,
    trips.duration_seconds
FROM
    trips
    INNER JOIN weather_records ON trips.start_ts = weather_records.ts
WHERE
    trips.pickup_location_id = 50
    AND trips.dropoff_location_id = 63
    AND EXTRACT(DOW FROM trips.start_ts) = 6
ORDER BY
    trips.trip_id;